# Exercise I: Exploratory Data Analysis (EDA)

## Learning objectives

By the end of this exercise, you should be able to:

- Inspect the dimensions and variables of a dataset
- Distinguish categorical and numerical variables
- Identify missing values and unusual observations
- Visualize univariate and multivariate distributions
- Detect potential confounding variables
- Formulate questions for subsequent predictive analysis


### Before we begin

This exercise uses data from the Autism Brain Imaging Data Exchange II (ABIDE II).

    
Before loading the data, answer:

1. What should one row represent?
2. Which variables do you expect to be categorical?
3. Which variables could confound a relationship between brain structure and diagnosis?


## 1. Importing and data loading

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

### Loading the ABIDE-II phenotypic data

The [Autism Brain Imaging Data Exchange II (ABIDE-II)](https://fcon_1000.projects.nitrc.org/indi/abide/abide_II.html) combines neuroimaging and phenotypic data collected across 19 international research sites. The complete phenotypic table contains several hundred variables, including demographic information, diagnostic assessments, cognitive scores, behavioral questionnaires, and acquisition-related variables.

The meaning and coding of each variable are documented in the official [ABIDE-II Phenotypic Data Legend](https://fcon_1000.projects.nitrc.org/indi/abide/ABIDEII_Data_Legend.pdf).

The complete ABIDE-II phenotypic dataset contains 348 variables. For this
exercise, we will use a curated subset of 39 demographic, diagnostic,
cognitive, and behavioral variables.

Some variables are recorded for nearly every participant, whereas others were collected only at particular sites or for particular participant groups.

In [7]:
import pandas as pd

PHENOTYPES_URL = (
    "https://raw.githubusercontent.com/"
    "neurohackademy/nh2020-curriculum/"
    "e4eed3c4daa7f40b0ba931182a8c7e5e691dba6b/"
    "tu-machine-learning-yarkoni/data/abide2_phenotypic.csv"
)

# Use a pre-selected subset of columns from the table:
COLUMNS_URL = "https://raw.githubusercontent.com/yoavmp/ml-neuro-tutorials/main/book/config/eda_phenotype_columns.json"

phenotypes = pd.read_csv(PHENOTYPES_URL, encoding="latin-1", low_memory=False)
phenotypes.columns = phenotypes.columns.str.strip()
phenotypes = phenotypes[pd.read_json(COLUMNS_URL, typ="series").tolist()].copy()

rows, cols = phenotypes.shape
print(f"Data table shape: {rows,cols}") 

Data table shape: (1114, 39)


## 2. Data inspection

Before calculating statistics or creating visualizations, it is useful to inspect several individual observations. This can reveal how the dataset is organized, how variables are represented, and whether anything appears unexpected.

Pandas provides several ways to inspect rows:

- `phenotypes.head()` displays the first rows.
- `phenotypes.tail()` displays the last rows.
- `phenotypes.sample()` displays randomly selected rows.
- Custom selection scripts can display particular rows, regularly spaced observations, or participants who meet specific conditions.

`head()` and `tail()` are convenient, but they may not represent the entire dataset. For example, datasets are sometimes sorted by acquisition site, diagnosis, age, or participant identifier. A random sample can provide a broader first impression.

### Inspecting a random sample

The `sample()` method selects rows randomly. Setting `random_state` makes the selection reproducible, meaning that everyone running the notebook will see the same participants.

In [12]:
phenotypes.sample(n=8, random_state=42)

,SITE_ID,SUB_ID,DX_GROUP,PDD_DSM_IV_TR,AGE_AT_SCAN,SEX,HANDEDNESS_CATEGORY,HANDEDNESS_SCORES,FIQ,VIQ,...,SRS_TOTAL_T,SCQ_TOTAL,RBSR_6SUBSCALE_TOTAL,MASC_TOTAL_T,BRIEF_BRI_T,BRIEF_MI_T,BRIEF_GEC_T,CBCL_6-18_INTERNAL_T,CBCL_6-18_EXTERNAL_T,CBCL_6-18_TOTAL_PROBLEM_T
879,ABIDEII-SDSU_1,28909,1,NaN,10.900000,1,1.0,80.0,104.0,105.0,...,87.0,18.0,26.0,NaN,65.0,71.0,70.0,NaN,NaN,NaN
101,ABIDEII-EMC_1,29907,2,NaN,6.685832,1,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1111,ABIDEII-USM_1,29525,2,0.0,23.290900,1,1.0,80.0,123.0,107.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
726,ABIDEII-OHSU_1,28988,1,NaN,11.000000,2,1.0,100.0,96.0,NaN,...,77.0,NaN,NaN,57.0,NaN,NaN,NaN,NaN,NaN,NaN
291,ABIDEII-IP_1,29600,2,0.0,46.430000,2,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
868,ABIDEII-SDSU_1,28890,1,NaN,15.200000,1,3.0,-40.0,80.0,75.0,...,94.0,26.0,49.0,NaN,84.0,71.0,78.0,NaN,NaN,NaN
951,ABIDEII-TCD_1,29100,1,2.0,18.750000,1,1.0,NaN,99.0,99.0,...,84.0,18.0,30.0,NaN,NaN,NaN,NaN,61.0,56.0,64.0
260,ABIDEII-IP_1,29595,1,1.0,27.360000,1,1.0,NaN,108.0,NaN,...,NaN,NaN,16.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Questions for discussion

Examine the sampled rows and the column names before continuing.

1. **What does each row represent?**

2. **Which variables are categorical?**  
   Remember that categorical variables are not necessarily stored as text. Some categories may be represented using numerical codes.

3. **Are there variables or subjects that seem problematic already?**

Answer before inspecting the complete data summary.

## 3. Statistical inspection

Looking at individual rows helps us understand the structure of a dataset, but it does not summarize the dataset as a whole. Pandas provides two useful starting points:

- `DataFrame.info()` summarizes the dataset's structure and data types.
- `DataFrame.describe()` calculates descriptive statistics for each variable.

These methods answer different questions and should usually be used together.

### Dataset structure with `info()`

`info()` reports:

- the number of rows and columns;
- each column's data type;
- the number of non-missing observations;
- an estimate of memory usage.

It is particularly useful for detecting variables with missing data and variables stored using an unexpected data type.

In [14]:
phenotypes.info()

<class 'pandas.DataFrame'>
RangeIndex: 1114 entries, 0 to 1113
Data columns (total 39 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   SITE_ID                    1114 non-null   str    
 1   SUB_ID                     1114 non-null   int64  
 2   DX_GROUP                   1114 non-null   int64  
 3   PDD_DSM_IV_TR              595 non-null    float64
 4   AGE_AT_SCAN                1114 non-null   float64
 5   SEX                        1114 non-null   int64  
 6   HANDEDNESS_CATEGORY        1091 non-null   float64
 7   HANDEDNESS_SCORES          641 non-null    float64
 8   FIQ                        1015 non-null   float64
 9   VIQ                        799 non-null    float64
 10  PIQ                        872 non-null    float64
 11  FIQ_TEST_TYPE              1015 non-null   str    
 12  CURRENT_MED_STATUS         991 non-null    float64
 13  EYE_STATUS_AT_SCAN         1113 non-null   float64
 14  ADI

#### Questions

Examine the output of `phenotypes.info()`.

1. How many participants and variables are present?
2. Which variables contain missing observations?
3. Are any categorical variables stored as numbers/strings?
4. Why might pandas assign the `object` data type to a column?

#### Interpreting `info()`

**Strengths**

- Provides a quick overview of the entire dataframe.
- Identifies columns containing missing values.
- Helps detect incorrect data types.
- Remains useful even when the dataframe has many rows.

**Limitations**

- Does not show distributions, ranges, or unusual values.
- Does not explain what numerical category codes mean.
- A column's storage type is not necessarily its statistical type. For example, diagnosis and sex may be stored as integers even though they are categorical variables.
- Reports the number of missing observations, but not whether missingness follows a meaningful pattern.

### Identifying hidden categorical variables

A dataframe's **storage type** is not necessarily the variable's **statistical type**.

For example, pandas initially reads:

- `SITE_ID` and `FIQ_TEST_TYPE` as text (`object`);
- `DX_GROUP` and `SEX` as integers;
- `ADOS_MODULE` and `CURRENT_MED_STATUS` as floating-point numbers because they contain missing values.

Nevertheless, all of these variables represent categories rather than numerical measurements. Their numerical codes are labels: arithmetic operations such as calculating their mean are generally not meaningful.

Before calculating descriptive statistics, we should explicitly tell pandas which variables are categorical.

#### Identify the variables

Using the column names and the [ABIDE-II Phenotypic Data Legend](https://fcon_1000.projects.nitrc.org/indi/abide/ABIDEII_Data_Legend.pdf), consider:

1. Which numerical columns represent category codes?
2. Which text columns represent a limited set of categories?
3. Should `SUB_ID` be considered a measurement, a category, or an identifier?
4. Is `ADOS_MODULE` an ordered severity scale, or does each number identify a different assessment module?

In [17]:
categorical_columns = [
    "SITE_ID",
    "DX_GROUP",
    "PDD_DSM_IV_TR",
    "SEX",
    "HANDEDNESS_CATEGORY",
    "FIQ_TEST_TYPE",
    "CURRENT_MED_STATUS",
    "EYE_STATUS_AT_SCAN",
    "ADOS_MODULE",
    "SRS_VERSION",
    "SRS_INFORMANT",
]

# Defining the dtype (data type) to be categorical:
phenotypes[categorical_columns] = phenotypes[categorical_columns].astype("category")

# Participant IDs are labels, not numerical measurements.
phenotypes["SUB_ID"] = phenotypes["SUB_ID"].astype("string")

The `category` dtype tells pandas that these columns contain a finite set of possible groups or labels. Missing values remain missing after the conversion.

The conversion does not change the meaning of the original codes. For example, the values `1` and `2` in `DX_GROUP` remain `1` and `2`; pandas now simply knows that they represent categories rather than quantities with a relative distance.

None of these variables are declared as ordered categories. In particular, the numerical values of `ADOS_MODULE` identify different assessment modules and should not be interpreted as increasing levels of autism severity (If you are not sure what the variable stands for - ask the researchers who collected the data! Or in this case - just google it!).

In [18]:
phenotypes[categorical_columns + ["SUB_ID"]].dtypes

SITE_ID                category
DX_GROUP               category
PDD_DSM_IV_TR          category
SEX                    category
HANDEDNESS_CATEGORY    category
FIQ_TEST_TYPE          category
CURRENT_MED_STATUS     category
EYE_STATUS_AT_SCAN     category
ADOS_MODULE            category
SRS_VERSION            category
SRS_INFORMANT          category
SUB_ID                   string
dtype: object

#### Why does this matter?

Correctly assigning categorical data types:

- prevents coded categories from appearing in numerical summaries;
- makes categorical summaries more informative;
- helps plotting libraries treat categories as distinct groups;
- makes the intended meaning of each variable explicit.

However, pandas' categorical dtype does not automatically prepare a variable for machine-learning models. Later, categorical predictors may require an encoding method such as one-hot encoding (later in the course).

### Numerical summaries with `describe()`

By default, `describe()` summarizes numerical columns. Its output includes:

- `count`: number of non-missing observations;
- `mean`: arithmetic mean;
- `std`: sample standard deviation;
- `min` and `max`: smallest and largest observed values;
- `25%`, `50%`, and `75%`: quartiles of the distribution.

Transposing the output (using `.T`) places variables in rows, which is often easier to read when the dataset contains many columns.

In [21]:
phenotypes.describe().T
# And for the categorical variables you can use: 
# phenotypes.describe(include=["category"]).T

,count,mean,std,min,25%,50%,75%,max
AGE_AT_SCAN,1114.0,14.864374,9.161864,5.128,9.296575,11.503593,18.0,64.0
HANDEDNESS_SCORES,641.0,70.937499,43.101969,-100.000,64.290000,83.000000,100.0,100.0
FIQ,1015.0,111.023645,15.482364,49.000,101.000000,112.000000,122.0,151.0
VIQ,799.0,112.143930,16.415442,45.000,101.000000,112.000000,124.0,156.0
PIQ,872.0,108.342890,15.989322,53.000,98.000000,109.000000,119.0,149.0
ADI_R_SOCIAL_TOTAL_A,302.0,18.894040,5.925509,0.000,15.000000,19.000000,23.0,30.0
ADI_R_VERBAL_TOTAL_BV,301.0,15.089701,4.714014,0.000,12.000000,15.000000,19.0,25.0
ADI_R_RRB_TOTAL_C,302.0,5.705298,2.481005,0.000,4.000000,6.000000,7.0,12.0
ADOS_G_TOTAL,347.0,9.536023,4.723664,0.000,7.000000,10.000000,13.0,23.0
ADOS_2_TOTAL,269.0,12.345725,4.219203,1.000,9.000000,12.000000,15.0,25.0


#### Interpreting `describe()`

**Strengths**

- Quickly summarizes the center, spread, and range of numerical variables.
- The `count` column helps identify variables with missing observations.
- Minimum and maximum values can reveal impossible or unexpected observations.
- Differences between the mean and median can suggest a skewed distribution.

**Limitations**

- Categorical variables (with `Dtpye = Category`) are excluded by default.
- Numerical category codes may be summarized as if they were quantitative measurements.
- Summary statistics can hide multimodal distributions, outliers, and differences between groups or acquisition sites.
- A small `count` may indicate structured missingness rather than random data loss.
- Plausible minimum and maximum values do not guarantee that all observations are valid.

#### Questions

Use the output above to answer the following questions.

1. Which variables have the most missing data?
2. Find one variable for which the mean and median differ noticeably. What might cause this difference?
3. Do any minimum or maximum values appear biologically or clinically surprising?
4. Why is calculating a mean for `DX_GROUP` or `SEX` usually not meaningful?
5. Could two variables have identical means and standard deviations but very different distributions?